# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
### 🛠️ What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

### 🚀 Bonus
- Improve routing
- Add logging
- Add more tools


In [1]:
import ast
import json
import math
import operator
import re
import time
import logging
from datetime import datetime

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logger = logging.getLogger("single_agent")

In [2]:
class SafeCalculator:
    def __init__(self):
        self.operators = {
            ast.Add: operator.add,
            ast.Sub: operator.sub,
            ast.Mult: operator.mul,
            ast.Div: operator.truediv,
            ast.Pow: operator.pow,
            ast.Mod: operator.mod,
            ast.FloorDiv: operator.floordiv
        }

        self.unary_operators = {
            ast.UAdd: operator.pos,
            ast.USub: operator.neg
        }

    def _evaluate(self, node):

        if isinstance(node, ast.Constant):
            if isinstance(node.value, (int, float)):
                if math.isfinite(float(node.value)):
                    return node.value
                raise ValueError("Non-finite number is not allowed")
            raise ValueError("Invalid constant")

        if isinstance(node, ast.BinOp):
            if type(node.op) not in self.operators:
                raise ValueError("Unsupported mathematical operator")

            left = self._evaluate(node.left)
            right = self._evaluate(node.right)

            if isinstance(node.op, (ast.Div, ast.FloorDiv, ast.Mod)) and right == 0:
                raise ZeroDivisionError("Division by zero")

            if isinstance(node.op, ast.Pow):
                if abs(right) > 100:
                    raise ValueError("Exponent is too large")

            return self.operators[type(node.op)](left, right)

        if isinstance(node, ast.UnaryOp):
            if type(node.op) not in self.unary_operators:
                raise ValueError("Unsupported unary operator")

            operand = self._evaluate(node.operand)

            return self.unary_operators[type(node.op)](operand)

        raise ValueError("Invalid mathematical expression")

    def calculate(self, expression: str) -> str:

        expression = expression.strip()

        if not expression:
            raise ValueError("Empty mathematical expression")

        if len(expression) > 200:
            raise ValueError("Expression is too long")

        try:
            tree = ast.parse(expression, mode="eval")
            result = self._evaluate(tree.body)

            if isinstance(result, float):
                if result.is_integer():
                    return str(int(result))

            return str(result)

        except Exception as e:
            raise ValueError(f"Calculation failed: {str(e)}")


calculator_tool = SafeCalculator()

## Calculator Tool Implementation

Here I implemented the calculator functionality. Instead of directly using Python's `eval()` function, I used an AST-based approach to allow only valid mathematical operations.

This makes the calculator safer because the user input is treated as a mathematical expression rather than arbitrary Python code.

The calculator also handles invalid expressions and errors such as division by zero.


In [4]:
def extract_keywords(text: str) -> list:

    if not isinstance(text, str):
        raise TypeError("Keyword input must be a string")

    text = text.strip()

    if not text:
        return []

    words = re.findall(r"[A-Za-z0-9]+", text.lower())

    stop_words = {
    "about",
    "from",
    "that",
    "this",
    "with",
    "into",
    "have",
    "been",
    "were",
    "they",
    "their",
    "there",
    "which",
    "what",
    "where",
    "when",
    "will",
    "is",
    "are",
    "the",
    "and",
    "for",
    "extract",
    "keywords"
}

    keywords = []

    for word in words:

        if len(word) > 4 and word not in stop_words:
            if word not in keywords:
                keywords.append(word)

    return keywords[:5]

## Keyword Extraction Tool

The keyword extractor takes a text input and identifies important words from it.

First, the text is converted to lowercase and words are extracted using regular expressions. Common words and instruction words such as `from`, `the`, `extract`, and `keywords` are removed using a stop-word list.

Duplicate words are also removed and the tool returns a maximum of five keywords.

In [5]:
print(extract_keywords(
    "Artificial Intelligence is transforming industries"
))

['artificial', 'intelligence', 'transforming', 'industries']


In [6]:
def general_response(query: str) -> str:

    query = query.strip()

    if not query:
        return "Please enter a valid query."

    responses = {
        "hello": "Hello! I am your Single-Agent Smart Assistant.",
        "hi": "Hi! How can I help you?",
        "who are you": "I am a rule-based single-agent assistant with tool routing.",
        "what is machine learning?":
            "Machine learning is a branch of artificial intelligence that enables systems to learn patterns from data and make predictions or decisions."
    }

    normalized = query.lower().strip()

    if normalized in responses:
        return responses[normalized]

    return (
        "I received your query successfully. "
        "This query does not require a specialized tool."
    )

In [7]:
class AgentState:

    def __init__(self, query):

        self.query = query
        self.normalized_query = query.strip().lower()

        self.intent = None
        self.tool = None
        self.result = None

        self.status = "initialized"

        self.start_time = time.perf_counter()
        self.end_time = None

        self.trajectory = []

    def add_step(self, step, details=None):

        self.trajectory.append({
            "step": step,
            "details": details,
            "timestamp": datetime.now().isoformat()
        })

    def finish(self):

        self.end_time = time.perf_counter()

    def execution_time_ms(self):

        if self.end_time is None:
            return None

        return round(
            (self.end_time - self.start_time) * 1000,
            3
        )

In [8]:
def create_response(response_type, result):

    allowed_types = {
        "calculation",
        "keywords",
        "general",
        "error"
    }

    if response_type not in allowed_types:
        response_type = "error"
        result = "Invalid response type"

    return {
        "type": response_type,
        "result": result
    }

In [29]:
def agent(query: str):

    state = AgentState(query)

    logger.info("Agent received query: %s", query)

    try:

        if not isinstance(query, str):

            state.status = "error"

            state.add_step(
                "input_validation",
                "Query must be a string"
            )

            state.finish()

            return create_response(
                "error",
                "Query must be a string"
            )

        if not query.strip():

            state.status = "error"

            state.add_step(
                "input_validation",
                "Empty query"
            )

            state.finish()

            return create_response(
                "error",
                "Query cannot be empty"
            )

        query_lower = query.lower().strip()

        state.add_step(
            "normalization",
            query_lower
        )

        if "calculate" in query_lower:

            state.intent = "calculation"
            state.tool = "calculator"

            state.add_step(
                "routing",
                "Calculator Tool selected"
            )

            expression = re.sub(
                r"\bcalculate\b",
                "",
                query,
                flags=re.IGNORECASE
            ).strip()

            if not expression:
                raise ValueError(
                    "No mathematical expression provided"
                )

            state.add_step(
                "tool_input",
                expression
            )

            result = calculator_tool.calculate(expression)

            state.result = result
            state.status = "success"

            state.add_step(
                "tool_execution",
                result
            )

            state.finish()

            logger.info(
                "Calculation completed successfully"
            )

            return create_response(
                "calculation",
                result
            )

        elif "keywords" in query_lower:

            state.intent = "keywords"
            state.tool = "keyword_extractor"

            state.add_step(
                "routing",
                "Keyword Extraction Tool selected"
            )

            text = re.sub(
                r"\bkeywords\b",
                "",
                query,
                flags=re.IGNORECASE
            ).strip()

            if not text:
                raise ValueError(
                    "No text provided for keyword extraction"
                )

            state.add_step(
                "tool_input",
                text
            )

            result = extract_keywords(text)

            state.result = result
            state.status = "success"

            state.add_step(
                "tool_execution",
                result
            )

            state.finish()

            logger.info(
                "Keyword extraction completed successfully"
            )

            return create_response(
                "keywords",
                result
            )

        else:

            state.intent = "general"
            state.tool = "general_response"

            state.add_step(
                "routing",
                "General Response Handler selected"
            )

            result = general_response(query)

            state.result = result
            state.status = "success"

            state.add_step(
                "general_response",
                result
            )

            state.finish()

            logger.info(
                "General query processed successfully"
            )

            return create_response(
                "general",
                result
            )

    except Exception as e:

        state.status = "error"

        state.add_step(
            "error_handler",
            str(e)
        )

        state.finish()

        logger.error(
            "Agent execution failed: %s",
            str(e)
        )

        return create_response(
            "error",
            str(e)
        )

## Agent Logic and Conditional Routing

The agent is the main part of the project. It receives the user's query and first validates and normalizes the input.

After converting the query to lowercase, conditional routing is used:

- If the query contains `calculate`, it is routed to the Calculator Tool.
- If the query contains `keywords`, it is routed to the Keyword Extraction Tool.
- All other valid queries are sent to the General Response Handler.

The agent returns every result using the same JSON structure:

{
    "type": "...",
    "result": "..."
}

Error cases are also handled using the `error` type.

In [30]:
import inspect

print(inspect.getsource(agent))

def agent(query: str):

    state = AgentState(query)

    logger.info("Agent received query: %s", query)

    try:

        if not isinstance(query, str):

            state.status = "error"

            state.add_step(
                "input_validation",
                "Query must be a string"
            )

            state.finish()

            return create_response(
                "error",
                "Query must be a string"
            )

        if not query.strip():

            state.status = "error"

            state.add_step(
                "input_validation",
                "Empty query"
            )

            state.finish()

            return create_response(
                "error",
                "Query cannot be empty"
            )

        query_lower = query.lower().strip()

        state.add_step(
            "normalization",
            query_lower
        )

        if "calculate" in query_lower:

            state.intent = "calculation"
        

In [31]:
response = agent("Calculate 20 + 5")

print(json.dumps(response, indent=2))

{
  "type": "calculation",
  "result": "25"
}


In [32]:
response = agent(
    "Extract keywords from Artificial Intelligence is transforming industries"
)

print(json.dumps(response, indent=2))

{
  "type": "keywords",
  "result": [
    "artificial",
    "intelligence",
    "transforming",
    "industries"
  ]
}


In [33]:
print(logger)

<Logger single_agent (WARNING)>


In [34]:
queries = [
    "Calculate 20 + 5",
    "Calculate 100 / 4",
    "Calculate 2 ** 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?",
    "Hello",
    "",
    "Calculate",
    "Keywords"
]

for q in queries:

    response = agent(q)

    print("Query:", q)
    print(
        json.dumps(
            response,
            indent=2,
            ensure_ascii=False
        )
    )

    print("-" * 60)

ERROR:single_agent:Agent execution failed: No mathematical expression provided
ERROR:single_agent:Agent execution failed: No text provided for keyword extraction


Query: Calculate 20 + 5
{
  "type": "calculation",
  "result": "25"
}
------------------------------------------------------------
Query: Calculate 100 / 4
{
  "type": "calculation",
  "result": "25"
}
------------------------------------------------------------
Query: Calculate 2 ** 5
{
  "type": "calculation",
  "result": "32"
}
------------------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
{
  "type": "keywords",
  "result": [
    "artificial",
    "intelligence",
    "transforming",
    "industries"
  ]
}
------------------------------------------------------------
Query: What is machine learning?
{
  "type": "general",
  "result": "Machine learning is a branch of artificial intelligence that enables systems to learn patterns from data and make predictions or decisions."
}
------------------------------------------------------------
Query: Hello
{
  "type": "general",
  "result": "Hello! I am your Single-

In [35]:
def validate_response(response):

    if not isinstance(response, dict):
        return False

    if "type" not in response:
        return False

    if "result" not in response:
        return False

    allowed_types = {
        "calculation",
        "keywords",
        "general",
        "error"
    }

    if response["type"] not in allowed_types:
        return False

    return True

In [36]:
test_cases = [
    {
        "query": "Calculate 20 + 5",
        "expected_type": "calculation"
    },
    {
        "query": "Calculate 100 / 4",
        "expected_type": "calculation"
    },
    {
        "query": "Extract keywords from Artificial Intelligence is transforming industries",
        "expected_type": "keywords"
    },
    {
        "query": "What is machine learning?",
        "expected_type": "general"
    },
    {
        "query": "Hello",
        "expected_type": "general"
    },
    {
        "query": "",
        "expected_type": "error"
    },
    {
        "query": "Calculate",
        "expected_type": "error"
    },
    {
        "query": "Keywords",
        "expected_type": "error"
    }
]

In [37]:
def run_validation_tests():

    passed = 0
    failed = 0

    print("=" * 70)
    print("SINGLE-AGENT VALIDATION TEST")
    print("=" * 70)

    for index, test in enumerate(test_cases, start=1):

        query = test["query"]
        expected = test["expected_type"]

        response = agent(query)

        structure_valid = validate_response(response)
        type_valid = response["type"] == expected

        test_passed = structure_valid and type_valid

        if test_passed:
            passed += 1
            status = "PASS"
        else:
            failed += 1
            status = "FAIL"

        print(f"\nTest {index}: {status}")
        print(f"Query: {query}")
        print(f"Expected: {expected}")

        print(
            "Response:",
            json.dumps(
                response,
                indent=2,
                ensure_ascii=False
            )
        )

    total = passed + failed

    accuracy = (
        passed / total * 100
        if total > 0
        else 0
    )

    print("\n" + "=" * 70)
    print("VALIDATION SUMMARY")
    print("=" * 70)

    print("Total Tests :", total)
    print("Passed      :", passed)
    print("Failed      :", failed)
    print("Accuracy    :", f"{accuracy:.2f}%")

    return {
        "total_tests": total,
        "passed": passed,
        "failed": failed,
        "accuracy": round(accuracy, 2)
    }

In [38]:
validation_result = run_validation_tests()

ERROR:single_agent:Agent execution failed: No mathematical expression provided
ERROR:single_agent:Agent execution failed: No text provided for keyword extraction


SINGLE-AGENT VALIDATION TEST

Test 1: PASS
Query: Calculate 20 + 5
Expected: calculation
Response: {
  "type": "calculation",
  "result": "25"
}

Test 2: PASS
Query: Calculate 100 / 4
Expected: calculation
Response: {
  "type": "calculation",
  "result": "25"
}

Test 3: PASS
Query: Extract keywords from Artificial Intelligence is transforming industries
Expected: keywords
Response: {
  "type": "keywords",
  "result": [
    "artificial",
    "intelligence",
    "transforming",
    "industries"
  ]
}

Test 4: PASS
Query: What is machine learning?
Expected: general
Response: {
  "type": "general",
  "result": "Machine learning is a branch of artificial intelligence that enables systems to learn patterns from data and make predictions or decisions."
}

Test 5: PASS
Query: Hello
Expected: general
Response: {
  "type": "general",
  "result": "Hello! I am your Single-Agent Smart Assistant."
}

Test 6: PASS
Query: 
Expected: error
Response: {
  "type": "error",
  "result": "Query cannot be emp

In [39]:
def inspect_trajectory(query):

    state = AgentState(query)

    print("=" * 70)
    print("TRAJECTORY EVALUATION")
    print("=" * 70)

    print("Input Query:", query)

    query_lower = query.lower().strip()

    state.add_step(
        "normalization",
        query_lower
    )

    if "calculate" in query_lower:
        state.intent = "calculation"
        state.tool = "calculator"

        state.add_step(
            "routing",
            "calculator"
        )

    elif "keywords" in query_lower:
        state.intent = "keywords"
        state.tool = "keyword_extractor"

        state.add_step(
            "routing",
            "keyword_extractor"
        )

    else:
        state.intent = "general"
        state.tool = "general_response"

        state.add_step(
            "routing",
            "general_response"
        )

    state.finish()

    print(
        json.dumps(
            {
                "query": state.query,
                "intent": state.intent,
                "tool": state.tool,
                "trajectory": state.trajectory
            },
            indent=2
        )
    )

In [40]:
inspect_trajectory(
    "Calculate 20 + 5"
)

TRAJECTORY EVALUATION
Input Query: Calculate 20 + 5
{
  "query": "Calculate 20 + 5",
  "intent": "calculation",
  "tool": "calculator",
  "trajectory": [
    {
      "step": "normalization",
      "details": "calculate 20 + 5",
      "timestamp": "2026-08-08T18:36:40.978087"
    },
    {
      "step": "routing",
      "details": "calculator",
      "timestamp": "2026-08-08T18:36:40.978098"
    }
  ]
}


In [41]:
def evaluate_agent(test_cases):

    total = len(test_cases)
    completed = 0
    errors = 0
    execution_times = []

    for test in test_cases:

        start = time.perf_counter()

        response = agent(test["query"])

        end = time.perf_counter()

        execution_times.append(
            (end - start) * 1000
        )

        if (
            validate_response(response)
            and response["type"] == test["expected_type"]
        ):
            completed += 1

        if response["type"] == "error":
            errors += 1

    completion_rate = (
        completed / total * 100
        if total else 0
    )

    error_rate = (
        errors / total * 100
        if total else 0
    )

    average_time = (
        sum(execution_times) / len(execution_times)
        if execution_times else 0
    )

    metrics = {
        "total_tasks": total,
        "completed_tasks": completed,
        "task_completion_rate": round(
            completion_rate, 2
        ),
        "errors": errors,
        "error_rate": round(
            error_rate, 2
        ),
        "average_execution_time_ms": round(
            average_time, 3
        )
    }

    print(
        json.dumps(
            metrics,
            indent=2
        )
    )

    return metrics

In [42]:
metrics = evaluate_agent(test_cases)

ERROR:single_agent:Agent execution failed: No mathematical expression provided
ERROR:single_agent:Agent execution failed: No text provided for keyword extraction


{
  "total_tasks": 8,
  "completed_tasks": 8,
  "task_completion_rate": 100.0,
  "errors": 3,
  "error_rate": 37.5,
  "average_execution_time_ms": 0.376
}


In [43]:
print("=" * 70)
print("SINGLE-AGENT SMART ASSISTANT")
print("Type 'exit' to stop")
print("=" * 70)

while True:

    user_input = input("\nEnter query: ")

    if user_input.strip().lower() == "exit":
        print("Agent session terminated.")
        break

    response = agent(user_input)

    print(
        json.dumps(
            response,
            indent=2,
            ensure_ascii=False
        )
    )

SINGLE-AGENT SMART ASSISTANT
Type 'exit' to stop

Enter query: keywords


ERROR:single_agent:Agent execution failed: No text provided for keyword extraction


{
  "type": "error",
  "result": "No text provided for keyword extraction"
}

Enter query: Calculate 20 + 5
{
  "type": "calculation",
  "result": "25"
}

Enter query: Extract keywords from Artificial Intelligence is transforming industries
{
  "type": "keywords",
  "result": [
    "artificial",
    "intelligence",
    "transforming",
    "industries"
  ]
}

Enter query: What is machine learning?
{
  "type": "general",
  "result": "Machine learning is a branch of artificial intelligence that enables systems to learn patterns from data and make predictions or decisions."
}

Enter query: exit
Agent session terminated.


## Interactive Mode

Finally, I added an interactive mode so that the user can continuously enter queries.

The agent processes each query and prints the structured JSON response. The loop continues until the user enters `exit`.

This allows the complete agent pipeline to be tested with real-time user input.

## 📦 Expected Output Format

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```

In [22]:
# 🧪 Test Cases

queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?"
]

for q in queries:
    print("Query:", q)
    print("Response:", agent(q))
    print("-" * 50)

Query: Calculate 20 + 5
Response: {'type': 'unknown', 'result': 'Not implemented'}
--------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
Response: {'type': 'unknown', 'result': 'Not implemented'}
--------------------------------------------------
Query: What is machine learning?
Response: {'type': 'unknown', 'result': 'Not implemented'}
--------------------------------------------------
